In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
ADAPTER_PATH = "./smollm-dog-lora/20260401_143510_final_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
).to("cpu")
base_model.eval()

ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
).to("cpu")

ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH).to("cpu")
ft_model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\DFS\Desktop\gitrepo\workshops\model-alignment-lab\.venv_x86\Lib\site-packages\peft\peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'bas

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 960, padding_idx=2)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=960, out_features=960, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=960, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=960, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [5]:
def run_prompt(model, tokenizer, prompt, max_new_tokens=180):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [6]:
prompt = """How do I teach my dog to stay?
Respond in JSON only using this schema:
command, difficulty, steps, common_mistake, session_length_minutes, reward_tip."""

base_out = run_prompt(base_model, tokenizer, prompt)

In [8]:
print(base_out)

{
  "command": "stay",
  "difficulty": "easy",
  "steps": [
    {
      "step": "Choose a comfortable and safe location for your dog to stay.",
      "reward_tip": "Choose a quiet, quiet, quiet place.",
      "common_mistake": "Not staying in the same location.",
      "session_length_minutes": 5,
      "reward_tip": "Reward your dog with a treat and praise.",
      "difficulty": "medium"
    }
  ]
}


In [9]:
prompt = "How to teach my dog how to stay, return JSON"

base_out = run_prompt(base_model, tokenizer, prompt)
ft_out = run_prompt(ft_model, tokenizer, prompt)

print("=== BASE ===")
print(base_out)


=== BASE ===
To teach your dog how to stay, return JSON, you can follow these steps:

1. Start by placing your dog in a safe and quiet area.

2. Start by having your dog sit. This is a basic command that your dog will learn quickly.

3. Have your dog sit and then have you walk away from them.

4. Have your dog stay in the same position as they were in when you first placed them in the safe area.

5. Repeat this process several times until your dog starts to associate staying in the safe area with the command "stay".

6. Once your dog is comfortable with the command "stay", you can start to move them around the room.

7. When you move your dog, have them return to the safe area.

8. Repeat this process several times until your dog starts to return to the safe area when


In [11]:
print(ft_out)

To teach your dog how to stay, return JSON, you can follow these steps:

1. Start by placing your dog in a safe and quiet area.

2. Start by having your dog sit. This is a basic command that your dog will learn quickly.

3. Have your dog sit and then have you walk away from them.

4. Have your dog stay in the same position as they were in when you first placed them in the safe area.

5. Repeat this process several times until your dog starts to associate staying in the safe area with the command "stay".

6. Once your dog is comfortable with the command "stay", you can start to move them around the room.

7. When you move your dog, have them return to the safe area.

8. Repeat this process several times until your dog starts to return to the safe area when


In [ ]:
prompts = [
    "How to teach my dog how to stay, return JSON",
    "How do I teach my dog to stay? Respond in JSON only.",
    """How do I teach my dog to stay?
Respond in JSON only using this schema:
command, difficulty, steps, common_mistake, session_length_minutes, reward_tip."""
]